In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Create working directory
import os
WORK_DIR = '/content/drive/MyDrive/SindhiLM'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/data', exist_ok=True)
os.makedirs(f'{WORK_DIR}/models', exist_ok=True)
os.makedirs(f'{WORK_DIR}/tokenizers', exist_ok=True)

print(f"Working directory created: {WORK_DIR}")

Mounted at /content/drive
Working directory created: /content/drive/MyDrive/SindhiLM


In [2]:
# Core NLP/ML libraries
!pip install -q transformers datasets tokenizers accelerate
!pip install -q huggingface_hub

# Data processing
!pip install -q pandas numpy scikit-learn
!pip install -q datasketch  # For deduplication (MinHash)

# Text processing for Perso-Arabic scripts
!pip install -q regex
!pip install -q ftfy  # Text fixing

# Utilities
!pip install -q tqdm matplotlib seaborn

print("✅ All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
✅ All packages installed!


In [3]:
# Fifth Cell - Check data location
import os

# List what's in your data folder
data_path = f'{WORK_DIR}/data'
print(f"Contents of {data_path}:")
for item in os.listdir(data_path):
    item_path = os.path.join(data_path, item)
    if os.path.isfile(item_path):
        size_mb = os.path.getsize(item_path) / (1024 * 1024)
        print(f"  📄 {item} ({size_mb:.1f} MB)")
    else:
        print(f"  📁 {item}/")
        # List subfolder contents
        for subitem in os.listdir(item_path):
            print(f"      {subitem}")

Contents of /content/drive/MyDrive/SindhiLM/data:
  📄 mehran_(مهراڻ 3.4 1982ع)_Sindhiadabiboard_final.checked_tokenized.json (1.4 MB)
  📄 mehran_(مهراڻ 3 2019ع)_Sindhiadabiboard_final.checked_tokenized.json (5.7 MB)
  📄 mehran_(مهراڻ 3.4 1998ع)_Sindhiadabiboard_final.checked_tokenized.json (1.9 MB)
  📄 mehran_(مهراڻ 3 2023ع)_Sindhiadabiboard_final.checked_tokenized.json (3.1 MB)
  📄 mehran_(مهراڻ 3.4 1963ع)_Sindhiadabiboard_final.checked_tokenized.json (2.2 MB)
  📄 mehran_(مهراڻ 3.4 1974ع)_Sindhiadabiboard_final.checked_tokenized.json (1.5 MB)
  📄 mehran_(مهراڻ 3 2024ع)_Sindhiadabiboard_final.checked_tokenized.json (5.1 MB)
  📄 mehran_(مهراڻ 3.4 1960ع)_Sindhiadabiboard_final.checked_tokenized.json (2.9 MB)
  📄 mehran_(مهراڻ 3.4 1966ع)_Sindhiadabiboard_final.checked_tokenized.json (1.7 MB)
  📄 mehran_(مهراڻ 3.4 2001ع)_Sindhiadabiboard_final.checked_tokenized.json (2.0 MB)
  📄 mehran_(مهراڻ 3.4 1958ع)_Sindhiadabiboard_final.checked_tokenized.json (0.1 MB)
  📄 mehran_(مهراڻ 3 2021ع)_Sindh

In [5]:
# Cell 1: Check the main text file
import os

WORK_DIR = '/content/drive/MyDrive/SindhiLM'
data_path = f'{WORK_DIR}/data'

# Check the main training file
train_file = f'{data_path}/train_sindhi_text.txt'
if os.path.exists(train_file):
    size_mb = os.path.getsize(train_file) / (1024 * 1024)
    print(f"✅ Found main training file: {size_mb:.1f} MB")

    # Count lines (documents)
    with open(train_file, 'r', encoding='utf-8') as f:
        line_count = sum(1 for _ in f)
    print(f"📄 Total documents: {line_count:,}")
else:
    print("❌ Main training file not found")

✅ Found main training file: 665.9 MB
📄 Total documents: 1,032,640


In [6]:
# Cell 2: Memory-efficient data loader
import json
import re
import unicodedata
from pathlib import Path
from collections import Counter

def load_json_files_batch(file_paths, batch_size=50):
    """
    Load JSON files in batches to manage memory
    """
    texts = []
    for i, file_path in enumerate(file_paths):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            # Extract text based on structure
            if isinstance(data, list):
                for entry in data:
                    if isinstance(entry, dict) and 'text' in entry:
                        texts.append(entry['text'])
                    elif isinstance(entry, str):
                        texts.append(entry)
            elif isinstance(data, dict):
                if 'text' in data:
                    texts.append(data['text'])
                elif 'data' in data:
                    for entry in data['data']:
                        if isinstance(entry, dict) and 'text' in entry:
                            texts.append(entry['text'])

            # Yield batch when size reached
            if len(texts) >= batch_size:
                yield texts
                texts = []

        except Exception as e:
            print(f"⚠️ Error loading {file_path}: {e}")
            continue

    # Yield remaining
    if texts:
        yield texts

def load_text_file_in_chunks(file_path, chunk_size=10000):
    """
    Load large text file in chunks
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        chunk = []
        for line in f:
            chunk.append(line.strip())
            if len(chunk) >= chunk_size:
                yield chunk
                chunk = []
        if chunk:
            yield chunk

In [7]:
# Cell 3: Sindhi text cleaning functions
import regex as re  # Use regex module for better Unicode support

class SindhiTextCleaner:
    def __init__(self):
        # Sindhi Unicode ranges
        self.sindhi_ranges = [
            r'\u0600-\u06FF',  # Arabic
            r'\u0750-\u077F',  # Arabic Supplement
            r'\u08A0-\u08FF',  # Arabic Extended-A
            r'\uFB50-\uFDFF',  # Arabic Presentation Forms-A
            r'\uFE70-\uFEFF',  # Arabic Presentation Forms-B
        ]

        # Compile regex patterns
        self.perso_arabic_pattern = re.compile(
            f'[{"".join(self.sindhi_ranges)}]+'
        )

        # Common Sindhi-specific characters that need normalization
        self.char_mappings = {
            # Normalize different forms of same character
            '\u064A': '\u064A',  # Yeh
            '\u06CC': '\u064A',  # Farsi Yeh -> Arabic Yeh
            '\u06D2': '\u064A',  # Yeh Barree -> Arabic Yeh
        }

        # Noise patterns
        self.noise_patterns = [
            r'http\S+',  # URLs
            r'www\.\S+',
            r'\S+@\S+',  # Emails
            r'\d{4,}',   # Long numbers (likely IDs)
            r'[^\w\s\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]',  # Non-Sindhi chars
        ]

    def normalize_unicode(self, text):
        """Normalize Unicode characters"""
        # NFC normalization
        text = unicodedata.normalize('NFC', text)

        # Apply character mappings
        for old, new in self.char_mappings.items():
            text = text.replace(old, new)

        return text

    def remove_noise(self, text):
        """Remove URLs, emails, and other noise"""
        for pattern in self.noise_patterns:
            text = re.sub(pattern, ' ', text)
        return text

    def normalize_whitespace(self, text):
        """Normalize whitespace"""
        # Replace multiple spaces/newlines with single space
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    def clean(self, text):
        """Full cleaning pipeline"""
        if not text or len(text.strip()) < 10:
            return None

        # Normalize Unicode
        text = self.normalize_unicode(text)

        # Remove noise
        text = self.remove_noise(text)

        # Normalize whitespace
        text = self.normalize_whitespace(text)

        # Filter: must contain significant Perso-Arabic text
        perso_arabic_chars = len(self.perso_arabic_pattern.findall(text))
        if perso_arabic_chars < 10:  # At least 10 Sindhi characters
            return None

        return text

# Initialize cleaner
cleaner = SindhiTextCleaner()
print("✅ Sindhi text cleaner initialized")

✅ Sindhi text cleaner initialized


In [8]:
# Cell 4: Process the main training file
import pickle
from tqdm import tqdm

def process_main_text_file(input_file, output_file, cleaner, max_docs=None):
    """
    Process the main text file and save cleaned documents
    """
    cleaned_docs = []
    total_processed = 0
    total_kept = 0

    print(f"🔄 Processing {input_file}...")

    for chunk in load_text_file_in_chunks(input_file, chunk_size=5000):
        for doc in chunk:
            total_processed += 1

            # Clean document
            cleaned = cleaner.clean(doc)
            if cleaned:
                cleaned_docs.append(cleaned)
                total_kept += 1

            # Save batch periodically
            if len(cleaned_docs) >= 10000:
                # Append to file
                with open(output_file, 'a', encoding='utf-8') as f:
                    for d in cleaned_docs:
                        f.write(d + '\n')
                cleaned_docs = []
                print(f"  💾 Saved batch. Total kept: {total_kept:,}")

            if max_docs and total_processed >= max_docs:
                break

        if max_docs and total_processed >= max_docs:
            break

    # Save remaining
    if cleaned_docs:
        with open(output_file, 'a', encoding='utf-8') as f:
            for d in cleaned_docs:
                f.write(d + '\n')

    print(f"✅ Processing complete!")
    print(f"   Processed: {total_processed:,}")
    print(f"   Kept: {total_kept:,} ({100*total_kept/total_processed:.1f}%)")

    return total_kept

# Process main file
input_file = f'{data_path}/train_sindhi_text.txt'
output_file = f'{WORK_DIR}/data/sindhi_cleaned.txt'

# Create empty output file
open(output_file, 'w').close()

# Process (limit to first 500k docs for initial test, remove limit for full processing)
total_docs = process_main_text_file(input_file, output_file, cleaner, max_docs=500000)

🔄 Processing /content/drive/MyDrive/SindhiLM/data/train_sindhi_text.txt...
  💾 Saved batch. Total kept: 10,000
  💾 Saved batch. Total kept: 20,000
  💾 Saved batch. Total kept: 30,000
  💾 Saved batch. Total kept: 40,000
  💾 Saved batch. Total kept: 50,000
  💾 Saved batch. Total kept: 60,000
  💾 Saved batch. Total kept: 70,000
  💾 Saved batch. Total kept: 80,000
  💾 Saved batch. Total kept: 90,000
  💾 Saved batch. Total kept: 100,000
  💾 Saved batch. Total kept: 110,000
  💾 Saved batch. Total kept: 120,000
  💾 Saved batch. Total kept: 130,000
  💾 Saved batch. Total kept: 140,000
  💾 Saved batch. Total kept: 150,000
  💾 Saved batch. Total kept: 160,000
  💾 Saved batch. Total kept: 170,000
  💾 Saved batch. Total kept: 180,000
  💾 Saved batch. Total kept: 190,000
  💾 Saved batch. Total kept: 200,000
  💾 Saved batch. Total kept: 210,000
  💾 Saved batch. Total kept: 220,000
  💾 Saved batch. Total kept: 230,000
  💾 Saved batch. Total kept: 240,000
  💾 Saved batch. Total kept: 250,000
  💾 Saved

In [9]:
# Cell 5: Process selected JSON files (high-quality sources)
import glob

def get_high_quality_json_files(data_path):
    """
    Select high-quality JSON files to process
    Prioritize: Wikipedia, Mehran, Novels, Literature
    """
    priority_patterns = [
        '*Wikipedia*',
        '*mehran*',
        '*Navel*',
        '*Literature*',
        '*Stories*',
        '*Folk_Litrature*',
        '*Articles*',
    ]

    selected_files = []
    for pattern in priority_patterns:
        files = glob.glob(f'{data_path}/{pattern}')
        # Sort by size, take largest ones
        files_with_size = [(f, os.path.getsize(f)) for f in files]
        files_with_size.sort(key=lambda x: x[1], reverse=True)

        # Take top 20 from each category
        selected_files.extend([f for f, s in files_with_size[:20]])

    return list(set(selected_files))  # Remove duplicates

# Get high-quality files
json_files = get_high_quality_json_files(data_path)
print(f"📁 Selected {len(json_files)} high-quality JSON files")

# Process JSON files
def process_json_files(json_files, output_file, cleaner):
    total_docs = 0
    kept_docs = 0

    for file_path in tqdm(json_files, desc="Processing JSON files"):
        try:
            for batch in load_json_files_batch([file_path], batch_size=100):
                cleaned_batch = []
                for text in batch:
                    total_docs += 1
                    cleaned = cleaner.clean(text)
                    if cleaned:
                        cleaned_batch.append(cleaned)
                        kept_docs += 1

                # Append to output
                if cleaned_batch:
                    with open(output_file, 'a', encoding='utf-8') as f:
                        for d in cleaned_batch:
                            f.write(d + '\n')

        except Exception as e:
            print(f"⚠️ Error with {file_path}: {e}")
            continue

    print(f"✅ JSON processing complete!")
    print(f"   Total: {total_docs:,}, Kept: {kept_docs:,}")
    return kept_docs

# Process JSON files (append to same output)
json_docs = process_json_files(json_files[:50], output_file, cleaner)  # Process first 50

📁 Selected 124 high-quality JSON files


Processing JSON files: 100%|██████████| 50/50 [01:29<00:00,  1.79s/it]

✅ JSON processing complete!
   Total: 0, Kept: 0


In [10]:
# Cell 6: Deduplication using MinHash LSH
from datasketch import MinHash, MinHashLSH
import hashlib

def deduplicate_corpus(input_file, output_file, threshold=0.85):
    """
    Remove near-duplicate documents using MinHash LSH
    """
    print(f"🔄 Deduplicating {input_file}...")

    # Initialize LSH
    lsh = MinHashLSH(threshold=threshold, num_perm=128)

    unique_docs = []
    duplicates = 0
    total = 0

    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            total += 1
            doc = line.strip()
            if not doc:
                continue

            # Create MinHash
            m = MinHash(num_perm=128)
            # Use word-level shingling
            words = doc.split()
            for word in words:
                m.update(word.encode('utf-8'))

            # Check for duplicates
            if not lsh.query(m):
                lsh.insert(f"doc_{total}", m)
                unique_docs.append(doc)
            else:
                duplicates += 1

            # Write batch periodically
            if len(unique_docs) >= 10000:
                with open(output_file, 'a', encoding='utf-8') as f_out:
                    for d in unique_docs:
                        f_out.write(d + '\n')
                unique_docs = []
                print(f"  Processed: {total:,}, Duplicates: {duplicates:,}")

    # Write remaining
    if unique_docs:
        with open(output_file, 'a', encoding='utf-8') as f_out:
            for d in unique_docs:
                f_out.write(d + '\n')

    print(f"✅ Deduplication complete!")
    print(f"   Total: {total:,}")
    print(f"   Unique: {total - duplicates:,}")
    print(f"   Duplicates removed: {duplicates:,} ({100*duplicates/total:.1f}%)")

    return total - duplicates

# Run deduplication
dedup_input = output_file
dedup_output = f'{WORK_DIR}/data/sindhi_deduped.txt'

# Create empty file
open(dedup_output, 'w').close()

unique_count = deduplicate_corpus(dedup_input, dedup_output, threshold=0.85)

🔄 Deduplicating /content/drive/MyDrive/SindhiLM/data/sindhi_cleaned.txt...
  Processed: 10,280, Duplicates: 280
  Processed: 21,460, Duplicates: 1,460
  Processed: 31,550, Duplicates: 1,550
  Processed: 41,581, Duplicates: 1,581
  Processed: 51,592, Duplicates: 1,592
  Processed: 61,882, Duplicates: 1,882
  Processed: 71,889, Duplicates: 1,889
  Processed: 81,894, Duplicates: 1,894
  Processed: 91,904, Duplicates: 1,904
  Processed: 101,914, Duplicates: 1,914
  Processed: 112,037, Duplicates: 2,037
  Processed: 122,041, Duplicates: 2,041
  Processed: 132,054, Duplicates: 2,054
  Processed: 143,246, Duplicates: 3,246
  Processed: 153,382, Duplicates: 3,382
  Processed: 164,795, Duplicates: 4,795
  Processed: 175,055, Duplicates: 5,055
  Processed: 185,584, Duplicates: 5,584
  Processed: 195,627, Duplicates: 5,627
  Processed: 205,771, Duplicates: 5,771
  Processed: 215,801, Duplicates: 5,801
  Processed: 226,588, Duplicates: 6,588
  Processed: 236,744, Duplicates: 6,744
  Processed: 247

In [11]:
# Cell 7: Create train/validation split and final statistics
import random

def create_train_val_split(input_file, train_file, val_file, val_ratio=0.05):
    """
    Split corpus into train and validation sets
    """
    print(f"🔄 Creating train/val split...")

    # Count total lines
    total_lines = sum(1 for _ in open(input_file, 'r', encoding='utf-8'))
    val_size = int(total_lines * val_ratio)

    # Randomly select validation indices
    val_indices = set(random.sample(range(total_lines), val_size))

    train_count = 0
    val_count = 0

    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(train_file, 'w', encoding='utf-8') as f_train, \
         open(val_file, 'w', encoding='utf-8') as f_val:

        for idx, line in enumerate(tqdm(f_in, total=total_lines)):
            if idx in val_indices:
                f_val.write(line)
                val_count += 1
            else:
                f_train.write(line)
                train_count += 1

    print(f"✅ Split complete!")
    print(f"   Train: {train_count:,}")
    print(f"   Validation: {val_count:,}")

    return train_count, val_count

# Create final split
final_train = f'{WORK_DIR}/data/train.txt'
final_val = f'{WORK_DIR}/data/val.txt'

train_count, val_count = create_train_val_split(dedup_output, final_train, final_val)

# Calculate final statistics
def calculate_corpus_stats(file_path):
    total_chars = 0
    total_words = 0
    total_lines = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            total_lines += 1
            total_chars += len(line)
            total_words += len(line.split())

    return {
        'lines': total_lines,
        'chars': total_chars,
        'words': total_words,
        'avg_length': total_chars / total_lines if total_lines > 0 else 0
    }

train_stats = calculate_corpus_stats(final_train)
val_stats = calculate_corpus_stats(final_val)

print(f"\n{'='*50}")
print("📊 FINAL CORPUS STATISTICS")
print(f"{'='*50}")
print(f"Training set:")
print(f"  Documents: {train_stats['lines']:,}")
print(f"  Characters: {train_stats['chars']:,}")
print(f"  Words (approx): {train_stats['words']:,}")
print(f"  Avg doc length: {train_stats['avg_length']:.0f} chars")
print(f"\nValidation set:")
print(f"  Documents: {val_stats['lines']:,}")
print(f"  Characters: {val_stats['chars']:,}")
print(f"  Words (approx): {val_stats['words']:,}")
print(f"\nTotal corpus size: {(train_stats['chars'] + val_stats['chars']) / 1e6:.1f}M characters")
print(f"Estimated tokens: {(train_stats['words'] + val_stats['words']) / 1e6:.1f}M words")

🔄 Creating train/val split...


100%|██████████| 409883/409883 [00:04<00:00, 91609.55it/s]


✅ Split complete!
   Train: 389,389
   Validation: 20,494

📊 FINAL CORPUS STATISTICS
Training set:
  Documents: 389,389
  Characters: 150,326,060
  Words (approx): 32,254,275
  Avg doc length: 386 chars

Validation set:
  Documents: 20,494
  Characters: 8,014,868
  Words (approx): 1,719,824

Total corpus size: 158.3M characters
Estimated tokens: 34.0M words


# Step 3: Tokenizer Training
3.1 Setup Tokenizer Training

In [12]:
# Cell 1: Install and import tokenizer libraries
!pip install -q tokenizers
!pip install -q transformers  # For saving tokenizer in HuggingFace format

from tokenizers import Tokenizer, models, pre_tokenizers, trainers, normalizers
from tokenizers.normalizers import NFD, StripAccents, Sequence
from tokenizers.pre_tokenizers import Whitespace, Punctuation, Sequence as PreSeq
from tokenizers.processors import TemplateProcessing
import json
import os

WORK_DIR = '/content/drive/MyDrive/SindhiLM'

In [13]:
# Cell 2: Define Sindhi tokenizer configuration
def create_sindhi_tokenizer_config():
    """
    Create tokenizer optimized for Sindhi Perso-Arabic script
    """

    # Initialize BPE model
    tokenizer = Tokenizer(models.BPE())

    # Normalization: Unicode NFC + handle Sindhi-specific characters
    tokenizer.normalizer = Sequence([
        normalizers.NFC(),  # Canonical composition
        normalizers.Strip(),  # Remove leading/trailing whitespace
    ])

    # Pre-tokenization: Split on whitespace and punctuation
    # Important for Perso-Arabic script to handle attached punctuation
    tokenizer.pre_tokenizer = PreSeq([
        Whitespace(),  # Split on whitespace
        Punctuation(),  # Split punctuation (.,!?،؛ etc.)
    ])

    # Special tokens
    special_tokens = ["<|endoftext|>", "<|pad|>", "<|unk|>", "<|s|>", "<|eot|>"]

    # Trainer configuration
    trainer = trainers.BpeTrainer(
        vocab_size=24000,  # Target vocabulary size (between UrduLM's 20k-32k)
        min_frequency=2,   # Minimum frequency for token
        special_tokens=special_tokens,
        show_progress=True,
        # Limit initial alphabet to speed up training
        initial_alphabet=[],  # Will be learned from data
    )

    return tokenizer, trainer

tokenizer, trainer = create_sindhi_tokenizer_config()
print("✅ Tokenizer configuration created")
print(f"   Target vocab size: 24,000")
print(f"   Special tokens: <|endoftext|>, <|pad|>, <|unk|>, <|s|>, <|eot|>")

✅ Tokenizer configuration created
   Target vocab size: 24,000
   Special tokens: <|endoftext|>, <|pad|>, <|unk|>, <|s|>, <|eot|>


In [15]:
# Cell 3: Train tokenizer (memory-efficient)
def train_tokenizer_batch(train_file, tokenizer, trainer, max_lines=500000):
    """
    Train tokenizer in batches to handle large files
    """
    print(f"🔄 Training tokenizer on {train_file}...")

    # Read file in chunks
    batch_size = 10000
    lines = []
    total_lines = 0

    with open(train_file, 'r', encoding='utf-8') as f:
        for line in f:
            lines.append(line.strip())
            total_lines += 1

            if len(lines) >= batch_size:
                # Train on batch
                tokenizer.train_from_iterator(lines, trainer=trainer)
                lines = []
                print(f"   Processed: {total_lines:,} lines")

            if max_lines and total_lines >= max_lines:
                break

        # Train on remaining lines
        if lines:
            tokenizer.train_from_iterator(lines, trainer=trainer)

    print(f"✅ Training complete! Processed {total_lines:,} lines")
    return tokenizer

# Train (use 400k docs for good coverage)
train_file = f'{WORK_DIR}/data/train.txt'
tokenizer = train_tokenizer_batch(train_file, tokenizer, trainer, max_lines=400000)

🔄 Training tokenizer on /content/drive/MyDrive/SindhiLM/data/train.txt...
   Processed: 10,000 lines
   Processed: 20,000 lines
   Processed: 30,000 lines
   Processed: 40,000 lines
   Processed: 50,000 lines
   Processed: 60,000 lines
   Processed: 70,000 lines
   Processed: 80,000 lines
   Processed: 90,000 lines
   Processed: 100,000 lines
   Processed: 110,000 lines
   Processed: 120,000 lines
   Processed: 130,000 lines
   Processed: 140,000 lines
   Processed: 150,000 lines
   Processed: 160,000 lines
   Processed: 170,000 lines
   Processed: 180,000 lines
   Processed: 190,000 lines
   Processed: 200,000 lines
   Processed: 210,000 lines
   Processed: 220,000 lines
   Processed: 230,000 lines
   Processed: 240,000 lines
   Processed: 250,000 lines
   Processed: 260,000 lines
   Processed: 270,000 lines
   Processed: 280,000 lines
   Processed: 290,000 lines
   Processed: 300,000 lines
   Processed: 310,000 lines
   Processed: 320,000 lines
   Processed: 330,000 lines
   Processe

In [16]:
# Cell 4: Configure post-processing and save
def configure_tokenizer(tokenizer):
    """
    Add post-processing template for GPT-style format
    """
    # Post-processing: Add <|endoftext|> at end
    tokenizer.post_processor = TemplateProcessing(
        single="$A <|endoftext|>",
        special_tokens=[
            ("<|endoftext|>", tokenizer.token_to_id("<|endoftext|>")),
        ],
    )

    # Enable truncation
    tokenizer.enable_truncation(max_length=512)

    # Enable padding
    tokenizer.enable_padding(
        pad_id=tokenizer.token_to_id("<|pad|>"),
        pad_token="<|pad|>",
        length=512
    )

    return tokenizer

tokenizer = configure_tokenizer(tokenizer)

# Get vocabulary info
vocab_size = len(tokenizer.get_vocab())
print(f"✅ Tokenizer configured")
print(f"   Actual vocab size: {vocab_size:,}")
print(f"   Pad token ID: {tokenizer.token_to_id('<|pad|>')}")
print(f"   EOS token ID: {tokenizer.token_to_id('<|endoftext|>')}")

✅ Tokenizer configured
   Actual vocab size: 24,000
   Pad token ID: 1
   EOS token ID: 0


In [17]:
# Cell 5: Save tokenizer in multiple formats
def save_tokenizer(tokenizer, save_dir):
    """
    Save tokenizer in Tokenizers and HuggingFace formats
    """
    os.makedirs(save_dir, exist_ok=True)

    # Save in tokenizers format (fast)
    tokenizer_path = f"{save_dir}/tokenizer.json"
    tokenizer.save(tokenizer_path)
    print(f"💾 Saved tokenizer.json to {tokenizer_path}")

    # Also save vocabulary as text file for inspection
    vocab = tokenizer.get_vocab()
    vocab_path = f"{save_dir}/vocab.txt"
    with open(vocab_path, 'w', encoding='utf-8') as f:
        # Sort by token ID
        sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
        for token, idx in sorted_vocab:
            f.write(f"{token}\n")
    print(f"💾 Saved vocab.txt to {vocab_path}")

    # Save config
    config = {
        "vocab_size": len(vocab),
        "special_tokens": {
            "pad": "<|pad|>",
            "eos": "<|endoftext|>",
            "unk": "<|unk|>",
        },
        "model_max_length": 512,
    }
    config_path = f"{save_dir}/tokenizer_config.json"
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=2)
    print(f"💾 Saved config to {config_path}")

    return tokenizer_path

tokenizer_dir = f'{WORK_DIR}/tokenizers/sindhi_bpe_24k'
tokenizer_path = save_tokenizer(tokenizer, tokenizer_dir)

💾 Saved tokenizer.json to /content/drive/MyDrive/SindhiLM/tokenizers/sindhi_bpe_24k/tokenizer.json
💾 Saved vocab.txt to /content/drive/MyDrive/SindhiLM/tokenizers/sindhi_bpe_24k/vocab.txt
💾 Saved config to /content/drive/MyDrive/SindhiLM/tokenizers/sindhi_bpe_24k/tokenizer_config.json


In [18]:
# Cell 6: Test tokenizer on sample texts
def test_tokenizer(tokenizer, test_texts):
    """
    Test tokenizer encoding/decoding
    """
    print(f"{'='*60}")
    print("🔍 TOKENIZER TESTS")
    print(f"{'='*60}")

    for i, text in enumerate(test_texts, 1):
        print(f"\n--- Test {i} ---")
        print(f"Input: {text[:100]}...")

        # Encode
        encoding = tokenizer.encode(text)
        tokens = encoding.tokens
        ids = encoding.ids

        print(f"Tokens ({len(tokens)}): {tokens[:15]}...")
        print(f"Token IDs: {ids[:15]}...")

        # Decode
        decoded = tokenizer.decode(ids)
        print(f"Decoded: {decoded[:100]}...")

        # Check if reversible
        is_reversible = (text.strip() == decoded.strip())
        print(f"Reversible: {'✅' if is_reversible else '⚠️'}")

    return True

# Test samples (mix of Sindhi text types)
test_samples = [
    "سنڌي ٻولي هڪ خوبصورت ٻولي آهي جيڪا پاڪستان ۽ هندستان ۾ ڳالهائي وڃي ٿي.",
    "شاهه عبداللطيف ڀٽائي سنڌ جي عظيم شاعر هو.",
    "مهراڻ رسالو سنڌي ادب جو اهم ذريعو آهي.",
    "پاڪستان زندہ باد!",
    "هي هڪ تجربو آهي.",
]

test_tokenizer(tokenizer, test_samples)

🔍 TOKENIZER TESTS

--- Test 1 ---
Input: سنڌي ٻولي هڪ خوبصورت ٻولي آهي جيڪا پاڪستان ۽ هندستان ۾ ڳالهائي وڃي ٿي....
Tokens (512): ['سنڌي', 'ٻولي', 'هڪ', 'خوبصورت', 'ٻولي', 'آ', 'هي', 'جيڪا', 'پاڪستان', '۽', 'هندستان', '۾', 'ڳالهائي', 'وڃي', 'ٿي']...
Token IDs: [506, 505, 3329, 2439, 505, 100, 609, 599, 511, 187, 9391, 188, 3535, 442, 270]...
Decoded: سنڌي ٻولي هڪ خوبصورت ٻولي آ هي جيڪا پاڪستان ۽ هندستان ۾ ڳالهائي وڃي ٿي...
Reversible: ⚠️

--- Test 2 ---
Input: شاهه عبداللطيف ڀٽائي سنڌ جي عظيم شاعر هو....
Tokens (512): ['شا', 'هه', 'عبداللطيف', 'ڀٽائي', 'سنڌ', 'جي', 'عظيم', 'شاعر', 'هو', '<|endoftext|>', '<|pad|>', '<|pad|>', '<|pad|>', '<|pad|>', '<|pad|>']...
Token IDs: [363, 7276, 3976, 4866, 299, 238, 1944, 1033, 496, 0, 1, 1, 1, 1, 1]...
Decoded: شا هه عبداللطيف ڀٽائي سنڌ جي عظيم شاعر هو...
Reversible: ⚠️

--- Test 3 ---
Input: مهراڻ رسالو سنڌي ادب جو اهم ذريعو آهي....
Tokens (512): ['م', 'هر', 'اڻ', 'رسالو', 'سنڌي', 'ادب', 'جو', 'ا', 'هم', 'ذريعو', 'آ', 'هي', '<|endoftext|>', '<|pa

True

In [19]:
# Cell 7: Calculate fertility (tokens per word) - key metric from UrduLM
def analyze_tokenizer_efficiency(tokenizer, sample_file, num_samples=10000):
    """
    Calculate fertility: average tokens per word
    Lower is better (more efficient tokenization)
    """
    total_tokens = 0
    total_words = 0
    total_chars = 0

    with open(sample_file, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= num_samples:
                break

            text = line.strip()
            if not text:
                continue

            # Count words (whitespace split)
            words = text.split()
            total_words += len(words)
            total_chars += len(text)

            # Count tokens
            encoding = tokenizer.encode(text)
            total_tokens += len(encoding.tokens)

    fertility = total_tokens / total_words if total_words > 0 else 0
    chars_per_token = total_chars / total_tokens if total_tokens > 0 else 0

    print(f"{'='*60}")
    print("📊 TOKENIZER EFFICIENCY METRICS")
    print(f"{'='*60}")
    print(f"Documents analyzed: {num_samples:,}")
    print(f"Total words: {total_words:,}")
    print(f"Total tokens: {total_tokens:,}")
    print(f"\n🎯 Fertility (tokens/word): {fertility:.2f}")
    print(f"   Lower is better (target: 1.5-2.5 for good tokenizers)")
    print(f"\n📏 Characters per token: {chars_per_token:.1f}")
    print(f"   Higher is better (more text per token)")

    # Compare with hypothetical mBERT
    mbert_fertility = 2.8  # Typical for multilingual BERT on Perso-Arabic
    improvement = ((mbert_fertility - fertility) / mbert_fertility) * 100

    print(f"\n📈 vs mBERT (~2.8): {improvement:.1f}% more efficient")

    return fertility

fertility = analyze_tokenizer_efficiency(tokenizer, train_file, num_samples=5000)

📊 TOKENIZER EFFICIENCY METRICS
Documents analyzed: 5,000
Total words: 650,701
Total tokens: 2,560,000

🎯 Fertility (tokens/word): 3.93
   Lower is better (target: 1.5-2.5 for good tokenizers)

📏 Characters per token: 1.2
   Higher is better (more text per token)

📈 vs mBERT (~2.8): -40.5% more efficient


In [20]:
# Cell 8: Analyze vocabulary composition
def analyze_vocabulary(tokenizer):
    """
    Analyze what types of tokens are in vocabulary
    """
    vocab = tokenizer.get_vocab()

    # Categorize tokens
    categories = {
        'single_char': 0,      # Single characters
        'sindhi_chars': 0,     # Pure Sindhi/Perso-Arabic
        'mixed': 0,            # Mixed scripts
        'punctuation': 0,      # Punctuation
        'special': 0,          # Special tokens
        'subword': 0,          # BPE subwords (with ##)
    }

    perso_arabic_pattern = re.compile(r'^[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]+$')
    punct_pattern = re.compile(r'^[^\w\s]+$')

    for token in vocab.keys():
        if token.startswith('<|') and token.endswith('|>'):
            categories['special'] += 1
        elif len(token) == 1:
            categories['single_char'] += 1
        elif perso_arabic_pattern.match(token):
            categories['sindhi_chars'] += 1
        elif punct_pattern.match(token):
            categories['punctuation'] += 1
        elif '##' in token or any(c in token for c in ['ـ', '̃', '̄']):
            categories['subword'] += 1
        else:
            categories['mixed'] += 1

    print(f"{'='*60}")
    print("📚 VOCABULARY COMPOSITION")
    print(f"{'='*60}")
    print(f"Total tokens: {len(vocab):,}")
    print()

    for cat, count in sorted(categories.items(), key=lambda x: x[1], reverse=True):
        pct = 100 * count / len(vocab)
        bar = '█' * int(pct / 2)
        print(f"{cat:15}: {count:6,} ({pct:5.1f}%) {bar}")

    # Show sample tokens
    print(f"\n🔤 Sample Sindhi tokens:")
    sindhi_tokens = [t for t in vocab.keys() if perso_arabic_pattern.match(t) and len(t) > 2]
    print(f"   {', '.join(sorted(sindhi_tokens)[:20])}")

    return categories

vocab_stats = analyze_vocabulary(tokenizer)

📚 VOCABULARY COMPOSITION
Total tokens: 24,000

sindhi_chars   : 23,466 ( 97.8%) ████████████████████████████████████████████████
mixed          :    295 (  1.2%) 
single_char    :    232 (  1.0%) 
special        :      5 (  0.0%) 
subword        :      2 (  0.0%) 
punctuation    :      0 (  0.0%) 

🔤 Sample Sindhi tokens:
   آءٌ, آءُ, آءِ, آؤٽرام, آئر, آئرش, آئرلينڊ, آئرن, آئس, آئل, آئلينڊ, آئلينڊز, آئن, آئنز, آئنسٽائن, آئون, آئوٽ, آئي, آئيس, آئيلينڊز


In [21]:
# Cell 9: Save tokenizer training metadata
tokenizer_metadata = {
    "vocab_size": len(tokenizer.get_vocab()),
    "fertility": fertility,
    "training_samples": 400000,
    "special_tokens": {
        "pad_token": "<|pad|>",
        "eos_token": "<|endoftext|>",
        "unk_token": "<|unk|>",
        "pad_token_id": tokenizer.token_to_id("<|pad|>"),
        "eos_token_id": tokenizer.token_to_id("<|endoftext|>"),
        "unk_token_id": tokenizer.token_to_id("<|unk|>"),
    },
    "model_max_length": 512,
    "files": {
        "tokenizer_json": f"{tokenizer_dir}/tokenizer.json",
        "vocab_txt": f"{tokenizer_dir}/vocab.txt",
    }
}

metadata_path = f"{tokenizer_dir}/tokenizer_metadata.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(tokenizer_metadata, f, indent=2, ensure_ascii=False)

print(f"💾 Tokenizer metadata saved to {metadata_path}")
print(f"\n{'='*60}")
print("🎉 STEP 3 COMPLETE!")
print(f"{'='*60}")
print(f"✅ Custom Sindhi BPE tokenizer trained")
print(f"✅ Vocabulary size: {tokenizer_metadata['vocab_size']:,}")
print(f"✅ Fertility: {fertility:.2f} tokens/word")
print(f"✅ Saved to: {tokenizer_dir}")
print(f"\n➡️ Next: Step 4 - Model Training!")

💾 Tokenizer metadata saved to /content/drive/MyDrive/SindhiLM/tokenizers/sindhi_bpe_24k/tokenizer_metadata.json

🎉 STEP 3 COMPLETE!
✅ Custom Sindhi BPE tokenizer trained
✅ Vocabulary size: 24,000
✅ Fertility: 3.93 tokens/word
✅ Saved to: /content/drive/MyDrive/SindhiLM/tokenizers/sindhi_bpe_24k

➡️ Next: Step 4 - Model Training!


#🚀 Step 4: Model Training (GPT-2 Architecture)

###4.1 Setup Model Configuration

In [22]:
# Cell 1: Install training libraries
!pip install -q transformers datasets accelerate
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

import torch
import torch.nn as nn
from transformers import (
    GPT2Config, GPT2LMHeadModel,
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
import json
import os
from pathlib import Path

WORK_DIR = '/content/drive/MyDrive/SindhiLM'
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

✅ PyTorch version: 2.10.0+cu128
✅ CUDA available: True
✅ GPU: Tesla T4


In [39]:
# Create HuggingFace tokenizer for data collator
from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=tokenizer_path,
    pad_token="<|pad|>",
    eos_token="<|endoftext|>",
    unk_token="<|unk|>",
)
print("✅ HuggingFace tokenizer wrapper created")

✅ HuggingFace tokenizer wrapper created


In [34]:
# Cell 3 (OPTIMIZED): Load dataset into memory for fast training
import torch
from datasets import Dataset

def load_dataset_fast(file_path, tokenizer, max_samples=50000, max_length=512):
    """
    Load dataset into memory once, then train from RAM
    Much faster than disk streaming!
    """
    print(f"🔄 Loading {max_samples} samples into RAM...")

    texts = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= max_samples:
                break
            text = line.strip()
            if len(text) > 20:  # Filter very short lines
                texts.append(text)

    print(f"✅ Loaded {len(texts)} texts")
    print(f"🔄 Tokenizing...")

    # Tokenize all at once
    encodings = []
    for text in texts:
        encoding = tokenizer.encode(text)
        ids = encoding.ids[:max_length]

        # Pad
        if len(ids) < max_length:
            ids = ids + [0] * (max_length - len(ids))

        encodings.append(ids)

    # Create dataset
    input_ids = torch.tensor([e[:-1] for e in encodings])  # Remove last token
    labels = torch.tensor([e[1:] for e in encodings])      # Shift left by 1

    dataset = Dataset.from_dict({
        'input_ids': input_ids,
        'labels': labels,
    })

    print(f"✅ Dataset ready: {len(dataset)} samples")
    return dataset

# Load datasets (50k train, 5k val - adjust based on your RAM)
train_dataset = load_dataset_fast(f"{WORK_DIR}/data/train.txt", tokenizer, max_samples=50000)
val_dataset = load_dataset_fast(f"{WORK_DIR}/data/val.txt", tokenizer, max_samples=5000)

print(f"\n{'='*60}")
print(f"✅ Fast datasets created!")
print(f"   Train: {len(train_dataset):,} samples")
print(f"   Val: {len(val_dataset):,} samples")
print(f"   RAM usage: ~{(len(train_dataset) + len(val_dataset)) * 512 * 4 / 1e6:.0f} MB")

🔄 Loading 50000 samples into RAM...
✅ Loaded 50000 texts
🔄 Tokenizing...
✅ Dataset ready: 50000 samples
🔄 Loading 5000 samples into RAM...
✅ Loaded 5000 texts
🔄 Tokenizing...
✅ Dataset ready: 5000 samples

✅ Fast datasets created!
   Train: 50,000 samples
   Val: 5,000 samples
   RAM usage: ~113 MB


In [35]:
# Cell 4: Setup training arguments (FULLY CORRECTED)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"{WORK_DIR}/models/sindhi_lm_50m",

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    warmup_steps=1000,
    max_steps=50000,

    # Evaluation (CORRECTED: evaluation_strategy -> eval_strategy)
    eval_strategy="steps",
    eval_steps=1000,
    per_device_eval_batch_size=8,

    # Logging & Saving
    logging_steps=100,
    save_steps=2000,
    save_total_limit=3,

    # Optimization
    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_torch",
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Colab survival
    report_to="none",
    dataloader_num_workers=0,
)

print("✅ Training arguments configured")
print(f"   Checkpoint dir: {training_args.output_dir}")

# Check for existing checkpoints
checkpoint_dir = Path(training_args.output_dir)
if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob("checkpoint-*"))
    if checkpoints:
        print(f"✅ Found {len(checkpoints)} existing checkpoints")
        print(f"   Latest: {sorted(checkpoints)[-1].name}")
        print(f"   Training will resume from last checkpoint")
    else:
        print(f"   No checkpoints found, starting fresh")
else:
    os.makedirs(training_args.output_dir, exist_ok=True)
    print(f"   Created new output directory")

✅ Training arguments configured
   Checkpoint dir: /content/drive/MyDrive/SindhiLM/models/sindhi_lm_50m
   No checkpoints found, starting fresh


In [40]:
# Cell 5 (UPDATED): Simple data collator for in-memory dataset
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=hf_tokenizer,  # Use HF tokenizer format
    mlm=False,  # For causal LM (GPT-style)
    pad_to_multiple_of=8,
)

print("✅ Data collator created")

✅ Data collator created


In [41]:
# Cell 6: Initialize trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("✅ Trainer initialized")
print(f"\n{'='*60}")
print("🚀 READY TO TRAIN!")
print(f"{'='*60}")
print(f"Model: GPT-2 Small ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")
print(f"Training samples: {len(train_dataset):,}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Max steps: {training_args.max_steps}")
print(f"Estimated time: ~3-4 hours on T4 GPU")
print(f"\n💡 Training will start in next cell...")

✅ Trainer initialized

🚀 READY TO TRAIN!
Model: GPT-2 Small (37.8M params)
Training samples: 50,000
Epochs: 3
Max steps: 50000
Estimated time: ~3-4 hours on T4 GPU

💡 Training will start in next cell...


In [ ]:
# Cell 7: Train!
print("🔄 Starting training... (This will take 3-4 hours)")
print("💡 Tip: If Colab disconnects, re-run from Cell 1 and training will resume from last checkpoint")
print("="*60)

# Check for existing checkpoints to resume
checkpoint_dir = Path(training_args.output_dir)
last_checkpoint = None

if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob("checkpoint-*"))
    if checkpoints:
        # Sort by checkpoint number
        checkpoints = sorted(checkpoints, key=lambda x: int(x.name.split('-')[-1]))
        last_checkpoint = str(checkpoints[-1])
        print(f"🔄 Resuming from: {last_checkpoint}")

# Start training
trainer.train(resume_from_checkpoint=last_checkpoint)

print("="*60)
print("✅ Training complete!")

🔄 Starting training... (This will take 3-4 hours)
💡 Tip: If Colab disconnects, re-run from Cell 1 and training will resume from last checkpoint


Step,Training Loss,Validation Loss
1000,6.386785,6.455245
2000,5.975470,6.077173
3000,5.708231,5.820512
4000,5.465586,5.686661
5000,5.359755,5.556193
6000,5.281623,5.466476
7000,5.076363,5.406958
8000,5.048966,5.339320
9000,4.986893,5.273490
10000,4.802651,5.251234


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]